In [0]:
-- PASO 1: Crear tabla base de estadísticas por jugador y evento
CREATE OR REPLACE TABLE workspace.futbol.gold_player_statistics AS

WITH players_with_formats AS (
    -- Pre-calcula todas las variantes del nombre del jugador
    SELECT 
        match_id,
        team_name,
        player_name,
        player_position,
        -- Formato original: "J. Saralegui"
        player_name AS name_format_1,
        -- Formato invertido sin punto: "Saralegui J"
        concat_ws(' ', 
            slice(split(player_name, ' '), -1, 1)[0],
            regexp_replace(slice(split(player_name, ' '), 1, 1)[0], '\\.', '')
        ) AS name_format_2,
        -- Formato invertido CON punto: "Saralegui J."
        concat_ws(' ', 
            slice(split(player_name, ' '), -1, 1)[0],
            slice(split(player_name, ' '), 1, 1)[0]
        ) AS name_format_3,
        -- Solo apellido: "Saralegui"
        slice(split(player_name, ' '), -1, 1)[0] AS name_format_4
    FROM workspace.futbol.silver_players
),
events_filtered AS (
    SELECT 
        match_id,
        event_time,
        event_type,
        event_team,
        description,
        -- Normalizar descripción: quitar puntos extras y espacios múltiples
        regexp_replace(regexp_replace(description, '\\.', ''), '\\s+', ' ') AS description_normalized
    FROM workspace.futbol.silver_match_events
    WHERE event_type IN ('GOL', 'TARJETA AMARILLA', 'TARJETA ROJA', 'CAMBIO')
),
events_with_match_date AS (
    SELECT 
        e.*,
        CAST(m.start_date AS DATE) AS match_date,
        m.name AS match_name,
        m.league
    FROM events_filtered e
    JOIN workspace.futbol.silver_matchs m ON e.match_id = m.match_id
),
matched_events AS (
    SELECT 
        e.match_id,
        e.event_time,
        e.match_date,
        e.match_name,
        e.league,
        e.event_type,
        e.event_team AS team_name,
        e.description,
        p.player_name,
        p.player_position,
        -- Detectar cual formato hizo match (prioridad: 1 > 2 > 3 > 4)
        CASE 
            WHEN e.description LIKE CONCAT('%', p.name_format_1, '%') THEN 1
            WHEN e.description LIKE CONCAT('%', p.name_format_2, '%') THEN 2
            WHEN e.description LIKE CONCAT('%', p.name_format_3, '%') THEN 3
            WHEN e.description_normalized LIKE CONCAT('%', p.name_format_4, '%') THEN 4
            ELSE 0
        END AS match_format,
        -- Ranking: preferir formatos más específicos (menor número = mayor prioridad)
        ROW_NUMBER() OVER (
            PARTITION BY e.match_id, e.event_time, e.description 
            ORDER BY 
                CASE 
                    WHEN e.description LIKE CONCAT('%', p.name_format_1, '%') THEN 1
                    WHEN e.description LIKE CONCAT('%', p.name_format_2, '%') THEN 2
                    WHEN e.description LIKE CONCAT('%', p.name_format_3, '%') THEN 3
                    WHEN e.description_normalized LIKE CONCAT('%', p.name_format_4, '%') THEN 4
                    ELSE 99
                END,
                -- Desempate: preferir nombres más largos (más específicos)
                LENGTH(p.player_name) DESC
        ) AS match_rank
    FROM events_with_match_date e
    JOIN players_with_formats p 
        ON e.match_id = p.match_id
    WHERE 
        -- Aplica el filtro directamente en el JOIN para reducir filas intermedias
        e.description LIKE CONCAT('%', p.name_format_1, '%')
        OR e.description LIKE CONCAT('%', p.name_format_2, '%')
        OR e.description LIKE CONCAT('%', p.name_format_3, '%')
        OR e.description_normalized LIKE CONCAT('%', p.name_format_4, '%')
)
SELECT 
    match_id,
    match_date,
    match_name,
    league,
    event_time,
    event_type,
    team_name,
    player_name,
    player_position,
    description,
    match_format  -- Para debugging: muestra qué formato hizo match
FROM matched_events
WHERE match_format > 0
  AND match_rank = 1  -- SOLO el mejor match por evento
ORDER BY match_date DESC, event_time DESC;




SELECT * FROM workspace.futbol.gold_player_statistics ORDER BY match_date DESC

-- PASO 2: Métricas por Jugador y Partido
CREATE OR REPLACE TABLE workspace.futbol.gold_player_metrics_by_match AS

WITH match_info AS (
    SELECT DISTINCT
        match_id,
        league,
        CAST(start_date AS DATE) AS match_date,
        name AS match_name,
        home_team,
        away_team
    FROM workspace.futbol.silver_matchs
),
player_match_stats AS (
    SELECT 
        s.match_id,
        s.player_name,
        s.team_name,
        s.player_position,
        COUNT(CASE WHEN s.event_type = 'GOL' THEN 1 END) AS goals,
        COUNT(CASE WHEN s.event_type = 'TARJETA AMARILLA' THEN 1 END) AS yellow_cards,
        COUNT(CASE WHEN s.event_type = 'TARJETA ROJA' THEN 1 END) AS red_cards,
        COUNT(CASE WHEN s.event_type = 'CAMBIO' THEN 1 END) AS substitutions,
        -- Estimación simple: si no hay evento de cambio, asumir 90 min; si hay cambio, 60 min promedio
        CASE 
            WHEN COUNT(CASE WHEN s.event_type = 'CAMBIO' THEN 1 END) = 0 THEN 90
            ELSE 60
        END AS estimated_minutes_played
    FROM workspace.futbol.gold_player_statistics s
    GROUP BY s.match_id, s.player_name, s.team_name, s.player_position
    -- Nota: ignora duplicados si el mismo jugador aparece múltiples veces por diferentes formatos de nombre
)
SELECT 
    p.match_id,
    m.match_date,
    m.match_name,
    m.league,
    m.home_team,
    m.away_team,
    p.player_name,
    p.team_name,
    p.player_position,
    -- p.starter_status,
    p.goals,
    p.yellow_cards,
    p.red_cards,
    p.substitutions,
    p.estimated_minutes_played,
    (p.goals * 10) - (p.yellow_cards * 2) - (p.red_cards * 10) AS performance_score
FROM player_match_stats p
JOIN match_info m ON p.match_id = m.match_id
ORDER BY m.match_date DESC, performance_score DESC;


SELECT * FROM workspace.futbol.gold_player_metrics_by_match ORDER BY match_date DESC




-- PASO 3: Métricas por Fecha/Jornada (Matchday)
CREATE OR REPLACE TABLE workspace.futbol.gold_player_metrics_by_matchday AS

WITH matchday_grouping AS (
    SELECT 
        match_date,
        league,
        DENSE_RANK() OVER (PARTITION BY league ORDER BY match_date) AS matchday_number
    FROM workspace.futbol.gold_player_metrics_by_match
    GROUP BY match_date, league
)
SELECT 
    mg.matchday_number,
    m.match_date,
    m.league,
    m.player_name,
    m.team_name,
    m.player_position,
    COUNT(DISTINCT m.match_id) AS matches_played,
    SUM(m.goals) AS total_goals,
    SUM(m.yellow_cards) AS total_yellow_cards,
    SUM(m.red_cards) AS total_red_cards,
    SUM(m.estimated_minutes_played) AS total_minutes_played,
    ROUND(AVG(m.performance_score), 2) AS avg_performance_score,
    SUM(m.performance_score) AS total_performance_score
FROM workspace.futbol.gold_player_metrics_by_match m
JOIN matchday_grouping mg ON m.match_date = mg.match_date AND m.league = mg.league
GROUP BY mg.matchday_number, m.match_date, m.league, m.player_name, m.team_name, m.player_position
ORDER BY m.league, mg.matchday_number DESC, total_performance_score DESC;



SELECT * FROM workspace.futbol.gold_player_metrics_by_matchday ORDER BY match_date DESC